In [ ]:
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession

# Create Spark Session
spark = SparkSession.builder \
    .appName("Task1_BigDataAnalysis") \
    .getOrCreate()

print("Spark Session Created Successfully!")


Spark Session Created Successfully!


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving Sample - Superstore.csv.zip to Sample - Superstore.csv (1).zip


In [ ]:
import zipfile
import os

zip_file_name = 'Sample - Superstore.csv (1).zip'
csv_file_name = 'Sample - Superstore.csv' # Assuming this is the name of the CSV file inside the zip

# Check if the zip file exists and extract it
if os.path.exists(zip_file_name):
    with zipfile.ZipFile(zip_file_name, 'r') as zip_ref:
        zip_ref.extractall('.') # Extract to the current directory
    print(f"'{zip_file_name}' extracted successfully.")
else:
    print(f"Error: Zip file '{zip_file_name}' not found.")

# Read CSV file using PySpark
df = spark.read.csv(
    csv_file_name,
    header=True,
    inferSchema=True
)

# Display first 5 rows
df.show(5)

'Sample - Superstore.csv (1).zip' extracted successfully.
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|F

In [ ]:
# Print the number of rows and columns
print("Total Rows:", df.count())
print("Total Columns:", len(df.columns))

# Display column names
print("\nColumn Names:")
print(df.columns)

# Display data types
print("\nData Types:")
df.printSchema()

Total Rows: 9994
Total Columns: 21

Column Names:
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']

Data Types:
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: str

In [ ]:
from pyspark.sql.functions import sum, avg, max, min, col, expr
from pyspark.sql.types import DoubleType

# Cast 'Sales' column to DoubleType, handling malformed values by setting them to NULL
# The error message suggests `try_cast`, which is more robust for messy data.
# Also, remove any leading/trailing spaces before casting.
df = df.withColumn("Sales", expr("try_cast(trim(Sales) AS DOUBLE)"))

# Total Sales
df.select(sum("Sales")).show()

# Average Sales
df.select(avg("Sales")).show()

# Maximum Sales
df.select(max("Sales")).show()

# Minimum Sales
df.select(min("Sales")).show()

+------------------+
|        sum(Sales)|
+------------------+
|2272449.8562999545|
+------------------+

+------------------+
|        avg(Sales)|
+------------------+
|234.41818199917006|
+------------------+

+----------+
|max(Sales)|
+----------+
|  22638.48|
+----------+

+----------+
|min(Sales)|
+----------+
|     0.444|
+----------+



In [ ]:
from pyspark.sql.functions import sum

# Total Sales by Category
category_sales = df.groupBy("Category").agg(
    sum("Sales").alias("Total Sales")
)

category_sales.show()

+---------------+-----------------+
|       Category|      Total Sales|
+---------------+-----------------+
|Office Supplies|703502.9280000031|
|      Furniture|733046.8612999996|
|     Technology|835900.0669999964|
+---------------+-----------------+



In [ ]:
from pyspark.sql.functions import sum

# Total Profit by Category
category_profit = df.groupBy("Category").agg(
    sum("Profit").alias("Total Profit")
)

category_profit.show()

+---------------+------------------+
|       Category|      Total Profit|
+---------------+------------------+
|Office Supplies|120632.87839999991|
|      Furniture| 19686.42720000003|
|     Technology|145388.29659999989|
+---------------+------------------+



In [ ]:
from pyspark.sql.functions import sum

# Top 10 Customers by Sales
top_customers = df.groupBy("Customer Name") \
    .agg(sum("Sales").alias("Total Sales")) \
    .orderBy("Total Sales", ascending=False)

top_customers.show(10)

+------------------+------------------+
|     Customer Name|       Total Sales|
+------------------+------------------+
|       Sean Miller|          25043.05|
|      Tamara Chand|19017.847999999998|
|      Raymond Buch|         15117.339|
|      Tom Ashbrook|          14595.62|
|     Adrian Barton|14355.610999999997|
|      Sanjit Chand|14142.333999999999|
|      Ken Lonsdale|         14071.917|
|      Hunter Lopez|12873.297999999999|
|      Sanjit Engle|12209.438000000002|
|Christopher Conant|         12129.072|
+------------------+------------------+
only showing top 10 rows


In [ ]:
from pyspark.sql.functions import sum

# Top 10 States by Sales
top_states = df.groupBy("State") \
    .agg(sum("Sales").alias("Total Sales")) \
    .orderBy("Total Sales", ascending=False)

top_states.show(10)

+------------+------------------+
|       State|       Total Sales|
+------------+------------------+
|  California| 450567.5915000007|
|    New York|309453.63299999974|
|       Texas|169553.63180000003|
|  Washington|136590.17199999996|
|Pennsylvania|114911.23700000002|
|     Florida|         88876.883|
|    Illinois|  79009.2869999999|
|        Ohio| 76617.64499999993|
|    Michigan| 75991.58400000002|
|    Virginia| 70309.08999999998|
+------------+------------------+
only showing top 10 rows


Final Business Insights
Dataset Overview
Total Rows: 9,994
Total Columns: 21
Sales Analysis
Total Sales: 2,272,449.86
Average Sales: 234.42
Maximum Sales: 22,638.48
Minimum Sales: 0.444
Category-wise Sales
Technology: 835,900.07
Furniture: 733,046.86
Office Supplies: 703,502.93

Insight: Technology generated the highest sales among all categories.

Category-wise Profit
Technology: 145,388.30
Office Supplies: 120,632.88
Furniture: 19,686.43

Insight: Although Furniture generated high sales, its profit was much lower than the other categories, indicating lower profit margins.

Top Customer
Sean Miller generated the highest sales (25,043.05).

Insight: High-value customers should be targeted with loyalty programs and personalized offers.

Top States by Sales
California – 450,567.59
New York – 309,453.63
Texas – 169,553.63

Insight: California is the company's strongest market and contributes the highest sales.